# 09 — Serie temporali IV: STL (1990) e il ponte NDSharp
Grafici PlotSharp source-build: costruire anche ../PlotSharp/src/PlotSharp/PlotSharp/PlotSharp.csproj -c Release. Runner da MathemaSharp: dotnet run --file ops/RunMigratedCoreNotebook.cs -- --id stl. Sei SVG e tabelle verificati; il ponte NDSharp resta separato dal renderer.
Kernel **JupyterNet** (C#) su runtime **.NET 11**. Prerequisito: `dotnet build src/MathemaSharp.slnx -c Release`. Da terminale: `scripts/run-notebooks.ps1` (imposta il roll-forward al runtime 11); in VS Code serve `DOTNET_ROLL_FORWARD=LatestMajor` e, finché .NET 11 è RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

**STL** (Cleveland, Cleveland, McRae & Terpenning, 1990) separa trend-ciclo, stagionalità e resto con LOESS. Si decompone prima, poi si prevede ogni componente per conto suo.

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using MathemaSharp.TimeSeries;
// Stagionalità che cresce nel tempo, trend non lineare, due outlier.
var rng = new Random(1990);
const int m = 12;
var n = 144;
var trendTrue = Enumerable.Range(0, n).Select(t => 50 + 10 * Math.Tanh((t - 70) / 30.0)).ToArray();
var seasonTrue = Enumerable.Range(0, n).Select(t => (3 + t / 48.0) * Math.Sin(2 * Math.PI * t / m)).ToArray();
var y = Enumerable.Range(0, n).Select(t => trendTrue[t] + seasonTrue[t] + 0.7 * Simulation.StandardNormal(rng)).ToArray();
y[40] += 15; y[100] -= 12;
Chart.Lines("Serie osservata (outlier a t = 40 e t = 100)", new Line("y", y));

Serie osservata (outlier a t = 40 e t = 100) 30 40 50 60 70 0 50 100 150 <path d='M64 169.23L68.21 164.31L72.43 161.1L76.64 155.21L80.85 156.21L85.07 159.79L89.28 167.66L93.49 171.66L97.71 178.58L101.92 178.91L106.13 177.43L110.35 174.19L114.56 172.29L118.77 162.61L122.99 156.1L127.2 157.78L131.41 153.15L135.63 157.03L139.84 169.88L144.05 172.61L148.27 180.41L152.48 184.71L156.69 178.61L160.91 175.02L165.12 165.63L169.33 161.47L173.55 148.28L177.76 151.6L181.97 155.16L186.19 156.16L190.4 161.93L194.61 174.65L198.83 176.37L203.04 177.39L207.25 173.8L211.47 168.82L215.68 164.09L219.89 156.26L224.11 146.11L228.32 146.38L232.53 85.53L236.75 149.87L240.96 158.54L245.17 169.43L249.39 170.74L253.6 169.2L257.81 162.27L262.03 164.35L266.24 154.33L270.45 146.59L274.67 142.06L278.88 137L283.09 134.91L287.31 139.53L291.52 146.5L295.73 161.76L299.95 159.11L304.16 163.91L308.37 162.43L312.59 149.45L316.8 134.13L321.01 128.69L325.23 126.92L329.44 121.57L333.65 121.66L337.87 123.8L342.08 133.87L346.29 146.71L350.51 152.9L354.72 149.12L358.93 146.8L363.15 136.22L367.36 123.88L371.57 113.44L375.79 111.46L380 107.27L384.21 104.1L388.43 110.27L392.64 118.79L396.85 128.58L401.07 131.23L405.28 129.55L409.49 129.39L413.71 126.95L417.92 113.14L422.13 98.61L426.35 89.34L430.56 92.76L434.77 92.25L438.99 97.69L443.2 98.99L447.41 109.26L451.63 120.95L455.84 123.98L460.05 116.62L464.27 114.35L468.48 101.05L472.69 87.24L476.91 84.09L481.12 83.73L485.33 128.05L489.55 85.82L493.76 104.62L497.97 101.12L502.19 111.05L506.4 117.11L510.61 109.76L514.83 102.27L519.04 94.06L523.25 80.95L527.47 78.84L531.68 73.14L535.89 72.89L540.11 80.71L544.32 94.64L548.53 103.95L552.75 110.24L556.96 111.98L561.17 111.12L565.39 96.08L569.6 89.16L573.81 85.4L578.03 71.98L582.24 65.11L586.45 72.58L590.67 75.42L594.88 90.79L599.09 107.38L603.31 107.83L607.52 106.51L611.73 107.24L615.95 99.36L620.16 87.74L624.37 79.49L628.59 73.79L632.8 69.25L637.01 74.53L641.23 82.07L645.44 89.04L649.65 104.47L653.87 109.66L658.08 111.97L662.29 115.86L666.51 101.34' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> y · x=0: 40.1 y · x=1: 41.3 y · x=2: 42.1 y · x=3: 43.5 y · x=4: 43.3 y · x=5: 42.4 y · x=6: 40.5 y · x=7: 39.5 y · x=8: 37.8 y · x=9: 37.7 y · x=10: 38 y · x=11: 38.8 y · x=12: 39.3 y · x=13: 41.7 y · x=14: 43.3 y · x=15: 42.9 y · x=16: 44 y · x=17: 43.1 y · x=18: 39.9 y · x=19: 39.2 y · x=20: 37.3 y · x=21: 36.2 y · x=22: 37.8 y · x=23: 38.6 y · x=24: 41 y · x=25: 42 y · x=26: 45.2 y · x=27: 44.4 y · x=28: 43.5 y · x=29: 43.3 y · x=30: 41.9 y · x=31: 38.7 y · x=32: 38.3 y · x=33: 38.1 y · x=34: 38.9 y · x=35: 40.2 y · x=36: 41.3 y · x=37: 43.3 y · x=38: 45.8 y · x=39: 45.7 y · x=40: 60.7 y · x=41: 44.8 y · x=42: 42.7 y · x=43: 40 y · x=44: 39.7 y · x=45: 40.1 y · x=46: 41.8 y · x=47: 41.3 y · x=48: 43.7 y · x=49: 45.7 y · x=50: 46.8 y · x=51: 48 y · x=52: 48.5 y · x=53: 47.4 y · x=54: 45.7 y · x=55: 41.9 y · x=56: 42.6 y · x=57: 41.4 y · x=58: 41.7 y · x=59: 45 y · x=60: 48.7 y · x=61: 50.1 y · x=62: 50.5 y · x=63: 51.8 y · x=64: 51.8 y · x=65: 51.3 y · x=66: 48.8 y · x=67: 45.6 y · x=68: 44.1 y · x=69: 45 y · x=70: 45.6 y · x=71: 48.2 y · x=72: 51.3 y · x=73: 53.8 y · x=74: 54.3 y · x=75: 55.4 y · x=76: 56.1 y · x=77: 54.6 y · x=78: 52.5 y · x=79: 50.1 y · x=80: 49.4 y · x=81: 49.9 y · x=82: 49.9 y · x=83: 50.5 y · x=84: 53.9 y · x=85: 57.5 y · x=86: 59.8 y · x=87: 58.9 y · x=88: 59.1 y · x=89: 57.7 y · x=90: 57.4 y · x=91: 54.9 y · x=92: 52 y · x=93: 51.2 y · x=94: 53.1 y · x=95: 53.6 y · x=96: 56.9 y · x=97: 60.3 y · x=98: 61.1 y · x=99: 61.2 y · x=100: 50.2 y · x=101: 60.7 y · x=102: 56 y · x=103: 56.9 y · x=104: 54.4 y · x=105: 52.9 y · x=106: 54.8 y · x=107: 56.6 y · x=108: 58.6 y · x=109: 61.9 y · x=110: 62.4 y · x=111: 63.8 y · x=112: 63.9 y · x=113: 61.9 y · x=114: 58.5 y · x=115: 56.2 y · x=116: 54.6 y · x=117: 54.2 y · x=118: 54.4 y · x=119: 58.1 y · x=120: 59.8 y · x=121: 60.8 y · x=122: 64.1 y · x=123: 65.8 y · x=124: 63.9 y · x=

## L'algoritmo
**LOESS** locale lineare: in ogni punto si stima una retta pesata sui $q$ vicini, con pesi tricubici $(1-(d/h)^3)^3$. Una retta viene riprodotta esattamente, anche quando si estrapola.

**Ciclo interno**, ripetuto $n_i$ volte:
1. detrend: $y - T$;
2. smoothing delle **sottoserie cicliche**: tutti i gennaio, tutti i febbraio, … ciascuno esteso di un ciclo prima e dopo;
3. **passa-basso** del risultato: MA($n_p$), MA($n_p$), MA(3), poi LOESS($n_l$);
4. $S = C - L$;
5. destagionalizza;
6. LOESS del trend con $n_t \ge 1.5n_p/(1-1.5/n_s)$.

**Ciclo esterno** (robustezza): pesi bisquare $B(|R_t|/6\,\mathrm{med}|R|)$ che declassano gli outlier nelle iterazioni successive.

Proprietà verificata dai test: su retta + stagionalità periodica a media nulla, STL separa le componenti **esattamente**. I LOESS riproducono le rette, e una media mobile lunga un periodo annulla la stagionalità.

In [ ]:
var plain = Stl.Decompose(y, m, seasonalSpan: 13);
var robust = Stl.Decompose(y, m, seasonalSpan: 13, innerIterations: 1, robustIterations: 6);
Chart.Panels("STL robusta: componenti", new Line("osservato", y), new Line("trend", robust.Trend), new Line("stagionalità", robust.Seasonal), new Line("resto", robust.Remainder));
double Rmse(double[] a, double[] b) => Accuracy.Rmse(a, b);
Table.Show(new[] { "STL", "RMSE trend vs vero", "RMSE stagionalità vs vera", "peso robusto t=40", "peso t=100" }, new[]
{
    new object[] { "non robusta", Rmse(plain.Trend, trendTrue), Rmse(plain.Seasonal, seasonTrue), plain.RobustnessWeights[40], plain.RobustnessWeights[100] },
    new object[] { "robusta (6 iterazioni)", Rmse(robust.Trend, trendTrue), Rmse(robust.Seasonal, seasonTrue), robust.RobustnessWeights[40], robust.RobustnessWeights[100] },
});

STL robusta: componenti osservato 30 40 50 60 70 0 50 100 150 <path d='M64 109.37L68.21 106.87L72.43 105.25L76.64 102.27L80.85 102.78L85.07 104.59L89.28 108.57L93.49 110.6L97.71 114.09L101.92 114.26L106.13 113.52L110.35 111.87L114.56 110.91L118.77 106.01L122.99 102.72L127.2 103.57L131.41 101.22L135.63 103.19L139.84 109.69L144.05 111.08L148.27 115.02L152.48 117.2L156.69 114.11L160.91 112.3L165.12 107.54L169.33 105.44L173.55 98.76L177.76 100.44L181.97 102.24L186.19 102.75L190.4 105.67L194.61 112.11L198.83 112.98L203.04 113.49L207.25 111.68L211.47 109.15L215.68 106.76L219.89 102.8L224.11 97.66L228.32 97.8L232.53 67L236.75 99.56L240.96 103.95L245.17 109.47L249.39 110.13L253.6 109.35L257.81 105.84L262.03 106.9L266.24 101.82L270.45 97.9L274.67 95.61L278.88 93.05L283.09 91.99L287.31 94.33L291.52 97.86L295.73 105.58L299.95 104.24L304.16 106.67L308.37 105.92L312.59 99.35L316.8 91.6L321.01 88.85L325.23 87.95L329.44 85.24L333.65 85.28L337.87 86.37L342.08 91.47L346.29 97.96L350.51 101.1L354.72 99.18L358.93 98.01L363.15 92.65L367.36 86.41L371.57 81.13L375.79 80.12L380 78L384.21 76.4L388.43 79.52L392.64 83.83L396.85 88.79L401.07 90.13L405.28 89.28L409.49 89.2L413.71 87.96L417.92 80.97L422.13 73.62L426.35 68.92L430.56 70.66L434.77 70.4L438.99 73.15L443.2 73.81L447.41 79.01L451.63 84.93L455.84 86.46L460.05 82.73L464.27 81.58L468.48 74.85L472.69 67.86L476.91 66.27L481.12 66.09L485.33 88.52L489.55 67.14L493.76 76.66L497.97 74.89L502.19 79.91L506.4 82.98L510.61 79.26L514.83 75.47L519.04 71.31L523.25 64.68L527.47 63.61L531.68 60.73L535.89 60.6L540.11 64.56L544.32 71.61L548.53 76.32L552.75 79.5L556.96 80.38L561.17 79.95L565.39 72.34L569.6 68.83L573.81 66.93L578.03 60.14L582.24 56.66L586.45 60.44L590.67 61.88L594.88 69.66L599.09 78.06L603.31 78.29L607.52 77.61L611.73 77.98L615.95 74L620.16 68.12L624.37 63.94L628.59 61.05L632.8 58.76L637.01 61.43L641.23 65.24L645.44 68.77L649.65 76.58L653.87 79.21L658.08 80.38L662.29 82.35L666.51 75' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> osservato · x=0: 40.1 osservato · x=1: 41.3 osservato · x=2: 42.1 osservato · x=3: 43.5 osservato · x=4: 43.3 osservato · x=5: 42.4 osservato · x=6: 40.5 osservato · x=7: 39.5 osservato · x=8: 37.8 osservato · x=9: 37.7 osservato · x=10: 38 osservato · x=11: 38.8 osservato · x=12: 39.3 osservato · x=13: 41.7 osservato · x=14: 43.3 osservato · x=15: 42.9 osservato · x=16: 44 osservato · x=17: 43.1 osservato · x=18: 39.9 osservato · x=19: 39.2 osservato · x=20: 37.3 osservato · x=21: 36.2 osservato · x=22: 37.8 osservato · x=23: 38.6 osservato · x=24: 41 osservato · x=25: 42 osservato · x=26: 45.2 osservato · x=27: 44.4 osservato · x=28: 43.5 osservato · x=29: 43.3 osservato · x=30: 41.9 osservato · x=31: 38.7 osservato · x=32: 38.3 osservato · x=33: 38.1 osservato · x=34: 38.9 osservato · x=35: 40.2 osservato · x=36: 41.3 osservato · x=37: 43.3 osservato · x=38: 45.8 osservato · x=39: 45.7 osservato · x=40: 60.7 osservato · x=41: 44.8 osservato · x=42: 42.7 osservato · x=43: 40 osservato · x=44: 39.7 osservato · x=45: 40.1 osservato · x=46: 41.8 osservato · x=47: 41.3 osservato · x=48: 43.7 osservato · x=49: 45.7 osservato · x=50: 46.8 osservato · x=51: 48 osservato · x=52: 48.5 osservato · x=53: 47.4 osservato · x=54: 45.7 osservato · x=55: 41.9 osservato · x=56: 42.6 osservato · x=57: 41.4 osservato · x=58: 41.7 osservato · x=59: 45 osservato · x=60: 48.7 osservato · x=61: 50.1 osservato · x=62: 50.5 osservato · x=63: 51.8 osservato · x=64: 51.8 osservato · x=65: 51.3 osservato · x=66: 48.8 osservato · x=67: 45.6 osservato · x=68: 44.1 osservato · x=69: 45 osservato · x=70: 45.6 osservato · x=71: 48.2 osservato · x=72: 51.3 osservato · x=73: 53.8 osservato · x=74: 54.3 osservato · x=75: 55.4 osservato · x=76: 56.1 osservato · x=77: 54.6 osservato · x=78: 52.5 osservato · x=79: 50.1 osservato · x=80: 49.4 osservato · x=81: 49.9 osservato · x=82: 49.9 osservato · x=83: 50.5 osservato · x=84: 53.9 osservato · x=85: 57.5 osservat

STL,RMSE trend vs vero,RMSE stagionalità vs vera,peso robusto t=40,peso t=100
non robusta,0.4921,0.6869,1,1
robusta (6 iterazioni),0.2375,0.3939,0,0


## Decomporre prima, prevedere le componenti poi
- Trend destagionalizzato: Holt con damping.
- Stagionalità: seasonal naive applicata alla sola componente stagionale (l'ultimo ciclo stimato si ripete).
- Previsione = somma delle due.

Il confronto è con Holt-Winters e con il seasonal naive sulla serie grezza.

In [ ]:
double[] train = y[..132], test = y[132..];
var stl = Stl.Decompose(train, m, seasonalSpan: 13, innerIterations: 1, robustIterations: 6);
var seasonallyAdjusted = train.Select((v, t) => v - stl.Seasonal[t]).ToArray();
var trendF = ExponentialSmoothing.Holt(seasonallyAdjusted, 0.3, 0.1, 12, phi: 0.9).Forecast;
var seasonF = ReferenceForecasts.SeasonalNaive(stl.Seasonal, m, 12);
var stlF = trendF.Select((v, h) => v + seasonF[h]).ToArray();
var hwF = ExponentialSmoothing.HoltWinters(train, m, 0.3, 0.05, 0.2, 12, phi: 0.9).Forecast;
var snF = ReferenceForecasts.SeasonalNaive(train, m, 12);
Table.Show(new[] { "Metodo", "MASE (m=12)" }, new[]
{
    new object[] { "STL + damped Holt + seasonal naive", Accuracy.Mase(test, stlF, train, m) },
    new object[] { "Holt-Winters damped", Accuracy.Mase(test, hwF, train, m) },
    new object[] { "Seasonal naive", Accuracy.Mase(test, snF, train, m) },
});
Chart.Lines("Ultimi 3 anni e previsioni (asse x: t − 108)", new Line("osservato", y[108..]), new Line("STL + componenti", stlF, Start: 24, Dashed: true), new Line("Holt-Winters", hwF, Start: 24, Dashed: true));

Metodo,MASE (m=12)
STL + damped Holt + seasonal naive,0.3903
Holt-Winters damped,0.3369
Seasonal naive,0.3722


Ultimi 3 anni e previsioni (asse x: t − 108) 50 55 60 65 70 0 10 20 30 40 <path d='M64 147.88L79.8 124.58L95.6 120.83L111.4 110.7L127.2 110.25L143 124.15L158.8 148.92L174.6 165.46L190.4 176.64L206.2 179.74L222 178.21L237.8 151.47L253.6 139.17L269.4 132.48L285.2 108.62L301 96.42L316.8 109.7L332.6 114.75L348.4 142.08L364.2 171.56L380 172.37L395.8 170.01L411.6 171.31L427.4 157.31L443.2 136.65L459 121.99L474.8 111.85L490.6 103.77L506.4 113.16L522.2 126.56L538 138.96L553.8 166.39L569.6 175.61L585.4 179.73L601.2 186.64L617 160.83' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/> osservato · x=0: 58.6 osservato · x=1: 61.9 osservato · x=2: 62.4 osservato · x=3: 63.8 osservato · x=4: 63.9 osservato · x=5: 61.9 osservato · x=6: 58.5 osservato · x=7: 56.2 osservato · x=8: 54.6 osservato · x=9: 54.2 osservato · x=10: 54.4 osservato · x=11: 58.1 osservato · x=12: 59.8 osservato · x=13: 60.8 osservato · x=14: 64.1 osservato · x=15: 65.8 osservato · x=16: 63.9 osservato · x=17: 63.2 osservato · x=18: 59.4 osservato · x=19: 55.3 osservato · x=20: 55.2 osservato · x=21: 55.6 osservato · x=22: 55.4 osservato · x=23: 57.3 osservato · x=24: 60.2 osservato · x=25: 62.2 osservato · x=26: 63.6 osservato · x=27: 64.8 osservato · x=28: 63.4 osservato · x=29: 61.6 osservato · x=30: 59.9 osservato · x=31: 56.1 osservato · x=32: 54.8 osservato · x=33: 54.2 osservato · x=34: 53.2 osservato · x=35: 56.8 STL + componenti · x=24: 60.3 STL + componenti · x=25: 64.1 STL + componenti · x=26: 64.2 STL + componenti · x=27: 65.3 STL + componenti · x=28: 64.7 STL + componenti · x=29: 63.3 STL + componenti · x=30: 59.4 STL + componenti · x=31: 56.2 STL + componenti · x=32: 55.6 STL + componenti · x=33: 55.1 STL + componenti · x=34: 55.5 STL + componenti · x=35: 57.8 Holt-Winters · x=24: 59.8 Holt-Winters · x=25: 61.8 Holt-Winters · x=26: 63 Holt-Winters · x=27: 63.4 Holt-Winters · x=28: 61.9 Holt-Winters · x=29: 61.7 Holt-Winters · x=30: 58.6 Holt-Winters · x=31: 56.3 Holt-Winters · x=32: 55.1 Holt-Winters · x=33: 54.8 Holt-Winters · x=34: 55.5 Holt-Winters · x=35: 57.4 Holt-Winters STL + componenti osservato

## Il ponte NDSharp
`NDSharp` (dal repository PySharp) è un ndarray con semantica NumPy: dtype a runtime, viste strided, broadcasting. `Numbers` resta tipizzato, con Generic Math e `Span<T>`. Il pacchetto `MathemaSharp.NDSharp` converte fra i due mondi:
- `ToVector()` e `ToMatrix()` copiano in ordine C con cast del dtype;
- `ToNDArray()` avvolge un `double[]` **senza copia**.

Il flusso tipico: preparazione dei dati con NDSharp, algoritmo con MathemaSharp.

In [ ]:
#load "../shared/NDSharp.csx"
using NDSharp;
using MathemaSharp.NDSharp;
// Preparazione "alla NumPy": rampa + seno costruiti con le ufunc di NDSharp, poi STL di MathemaSharp.
var t = np.Arange(0L, 120L);
var series = np.Add(np.Multiply(t, NDArray.Scalar(0.1)), np.Sin(np.Multiply(t, NDArray.Scalar(2 * Math.PI / 12))));
Console.WriteLine($"NDArray dtype={series.DType}, shape=({string.Join(",", series.Shape)})");
var decomposition = Stl.Decompose(series.ToVector(), 12);
var trendNd = decomposition.Trend.ToNDArray();
Console.WriteLine($"Trend come NDArray: dtype={trendNd.DType}, pendenza media={(decomposition.Trend[^1] - decomposition.Trend[0]) / 119:F6} (vera 0.1)");

NDArray dtype=Float64, shape=(120)
Trend come NDArray: dtype=Float64, pendenza media=0.100000 (vera 0.1)

## Esperimenti
- `seasonalSpan` più grande rende la stagionalità più rigida: provare 7 e 35 e guardare la componente stagionale.
- Senza robustezza, gli outlier finiscono nella stagionalità di *tutti* i mesi omologhi: perché?
- Sostituire il seasonal naive sulla componente stagionale con Holt-Winters a γ piccolo.